# Feature Stability Over Time — Walk-Forward IC Analysis

**Problem:** A feature might work 2015–2020 but fail 2021–2025.

**Test:** Walk-forward IC stability — split the sample into non-overlapping yearly windows and compute Spearman IC(feature, fwd_return_H24) in each.

**Stability criterion:** `sign_consistency ≥ 0.70`  
If the IC sign flips in more than 30% of years, the feature is flagged as unstable.

**Features tested:** All four PREDICTIVE-rated features from CATALOG.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from src.data.loader import FXDataLoader
from src.features import (
    compute_log_returns, atr, causal_hp_trend, hp_trend_slope,
    ma_spread_on_trend, ma_crossover_signal, trend_deviation_from_ma,
    trend_strength, vol_zscore,
    FeatureStabilityTester,
)

plt.style.use('seaborn-v0_8-darkgrid')
print('Imports OK')

## 1. Load data and compute features

In [ ]:
loader = FXDataLoader('../data/raw')
df = loader.load('USDJPY_10yr_1h_dukascopy')
close = df['close']
high = df['high']
low = df['low']
log_ret = compute_log_returns(close)
atr_s = atr(high, low, close, window=20)

print('Computing causal HP trend (may take ~60s)...')
trend = causal_hp_trend(close, lamb=3.9e9, window=500)
slope = hp_trend_slope(trend)
print('Done.')

features = {
    'ma_spread_on_trend': ma_spread_on_trend(trend, t1=72, t2=240, atr_series=atr_s),
    'trend_deviation_from_ma': trend_deviation_from_ma(trend, window=240, atr_series=atr_s),
    'ma_crossover_signal': ma_crossover_signal(trend, t1=72, t2=240).replace({np.nan: 0.0}),
    'trend_strength': trend_strength(trend, atr_s),
}

print(f'Loaded {len(df):,} bars, {len(features)} features prepared.')

## 2. Run walk-forward stability tests

In [ ]:
HORIZON = 24           # H24 = 24-bar (1 day) forward IC
PERIOD_BARS = 252 * 24  # 1 year of H1 bars

tester = FeatureStabilityTester(close, horizon=HORIZON, period_bars=PERIOD_BARS)
summary_df = tester.test_all(features)
print('Stability summary:')
summary_df.to_string(index=False)

In [ ]:
summary_df

## 3. IC by year bar charts

In [ ]:
results = {name: tester.test(series, name) for name, series in features.items()}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, (name, result) in zip(axes, results.items()):
    if not result.period_results:
        ax.set_title(f'{name}\n(no valid periods)')
        continue

    years = [p.period_start.year for p in result.period_results]
    ics = [p.ic for p in result.period_results]
    pvals = [p.p_value for p in result.period_results]

    colors = ['steelblue' if (v * result.mean_ic > 0) else 'tomato' for v in ics]
    bars = ax.bar(years, ics, color=colors, edgecolor='white', width=0.7)

    # Mark significant bars
    for bar, pv in zip(bars, pvals):
        if pv < 0.05:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.003 * (-1 if bar.get_height() < 0 else 1),
                '*', ha='center', fontsize=11, color='black',
            )

    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.axhline(result.mean_ic, color='orange', linewidth=1.5, linestyle='-', label=f'mean={result.mean_ic:.3f}')

    stable_label = '✅ STABLE' if result.is_stable else '⚠️ UNSTABLE'
    ax.set_title(f'{name}\n{stable_label}  sign_consistency={result.sign_consistency:.0%}', fontsize=10)
    ax.set_xlabel('Year')
    ax.set_ylabel('IC (Spearman ρ)')
    ax.legend(fontsize=9)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.suptitle(f'Walk-Forward IC by Year  (H={HORIZON} bars)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../reports/feature_stability_by_year.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: reports/feature_stability_by_year.png')

## 4. Stability verdict

In [ ]:
print('=== FEATURE STABILITY VERDICT ===')
for name, result in results.items():
    label = '✅ STABLE   ' if result.is_stable else '⚠️  UNSTABLE '
    print(f'{label} | {name:30s} | {result.stability_notes}')

print()
stable_count = sum(1 for r in results.values() if r.is_stable)
print(f'{stable_count}/{len(results)} features pass stability criterion (sign_consistency ≥ 0.70)')